# Step 1: Import Modules


In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from pathlib import Path

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "notebooks").is_dir() and (path / "data").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")

CLEANED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "edu_growth_cleaned.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


# Step 2: Settings and Variables


In [ ]:
f_in=CLEANED_DATA_PATH
f_out=ARTIFACTS_DIR / "mentor_assignments.csv"
subs=["coa","maths4","dstl","ds","python","cyber"]
labs=["ds","python","coa","cyber"]
bad=40
good=75


# Step 3: Load Data


In [ ]:
df=pd.read_csv(f_in)
print(df.shape)


# Step 4: Check All Columns


In [ ]:
c1=["roll_no","full_name","class_section"]
c2=["overall_attendance_pct","previous_cgpa"]
for s in subs:
    c2.append(s+"_st1_marks")
    c2.append(s+"_st2_marks")
    c2.append(s+"_put_marks")
    for u in range(1,6):
        c2.append(s+"_unit_"+str(u)+"_marks")
for l in labs:
    c2.append("lab_"+l+"_execution_score")
    c2.append("lab_"+l+"_viva_score")
miss=[]
for x in c1+c2:
    if x not in df.columns:
        miss.append(x)
print(miss)


# Step 5: Fix Data Formats


In [ ]:
for x in c2:
    df[x]=pd.to_numeric(df[x],errors="coerce").fillna(0)
df["class_section"]=df["class_section"].astype(str).str.strip().str.upper()


# Step 6: Find Maximum Marks


In [ ]:
a1=[]
for s in subs:
    a1.append(s+"_st1_marks")
a2=[]
for s in subs:
    a2.append(s+"_st2_marks")
a3=[]
for s in subs:
    a3.append(s+"_put_marks")
a4=[]
for s in subs:
    for u in range(1,6):
        a4.append(s+"_unit_"+str(u)+"_marks")
a5=[]
for l in labs:
    a5.append("lab_"+l+"_execution_score")
a6=[]
for l in labs:
    a6.append("lab_"+l+"_viva_score")
m1=df[a1].max().max()
m2=df[a2].max().max()
m3=df[a3].max().max()
m4=df[a4].max().max()
m5=df[a5].max().max()
m6=df[a6].max().max()


# Step 7: Calculate Subject Percentages


In [ ]:
for s in subs:
    v1=df[s+"_st1_marks"]/m1*100
    v2=df[s+"_st2_marks"]/m2*100
    v3=df[s+"_put_marks"]/m3*100
    tmp=[]
    for u in range(1,6):
        tmp.append(s+"_unit_"+str(u)+"_marks")
    v4=df[tmp].mean(axis=1)/m4*100
    df[s+"_pct"]=(v1+v2+v3+v4)/4


# Step 8: Calculate Lab Percentages


In [ ]:
for l in labs:
    v5=df["lab_"+l+"_execution_score"]/m5*100
    v6=df["lab_"+l+"_viva_score"]/m6*100
    df["lab_"+l+"_pct"]=(v5+v6)/2


# Step 9: Find Weak Subjects


In [ ]:
for s in subs:
    df[s+"_bad"]=(df[s+"_pct"]<bad).astype(int)


# Step 10: Find Weak Labs


In [ ]:
for l in labs:
    df["lab_"+l+"_bad"]=(df["lab_"+l+"_pct"]<bad).astype(int)


# Step 11: Count Total Weak


In [ ]:
b1=[]
for s in subs:
    b1.append(s+"_bad")
b2=[]
for l in labs:
    b2.append("lab_"+l+"_bad")
df["bad_s"]=df[b1].sum(axis=1)
df["bad_l"]=df[b2].sum(axis=1)
df["bad_tot"]=df["bad_s"]+df["bad_l"]
p1=[]
for s in subs:
    p1.append(s+"_pct")
df["avg_p"]=df[p1].mean(axis=1)
df["min_p"]=df[p1].min(axis=1)


# Step 12: Calculate Progress


In [ ]:
df["prog"]=(df[a2].mean(axis=1)/m2*100)-(df[a1].mean(axis=1)/m1*100)


# Step 13: KMeans Clustering


In [ ]:
f_dat=["avg_p","min_p","bad_s","bad_l","overall_attendance_pct","previous_cgpa","prog"]
sc=StandardScaler()
x_dat=sc.fit_transform(df[f_dat])
km=KMeans(n_clusters=4,random_state=42,n_init=10)
df["grp"]=km.fit_predict(x_dat)
df_grp=df.groupby("grp")[f_dat].mean().round(1)


# Step 14: Assign Group Names


In [ ]:
low=int(df_grp["avg_p"].idxmin())
high=int(df_grp["avg_p"].idxmax())
mid=[]
for x in df_grp.index:
    if x!=low and x!=high:
        mid.append(int(x))
if df_grp.loc[mid[0],"min_p"]<df_grp.loc[mid[1],"min_p"]:
    m1=mid[0]
    m2=mid[1]
else:
    m1=mid[1]
    m2=mid[0]
d1={low:"Need Help",m1:"Fell Down",m2:"Normal",high:"Topper"}
d2={"Need Help":1,"Fell Down":2,"Normal":3,"Topper":4}
df["g_name"]=df["grp"].map(d1)
df["prio"]=df["g_name"].map(d2)


# Step 15: Teacher Data


In [ ]:
t_dict={}
t_dict["coa"]={"Dr. Sharma":["CSE-A","CSE-B","IT-A"],"Prof. Verma":["CSE-C","IT-B"],"Dr. Ananya":["CS-AI","CS-DS"]}
t_dict["maths4"]={"Dr. Rajesh":["CSE-A","CSE-C","CS-AI"],"Prof. Sneha":["CSE-B","IT-B"],"Dr. Mehta":["IT-A","CS-DS"]}
t_dict["dstl"]={"Prof. Gupta":["CSE-A","IT-B"],"Dr. Iyer":["CSE-B","CSE-C","CS-DS"],"Prof. Khan":["IT-A","CS-AI"]}
t_dict["ds"]={"Dr. Singhal":["CSE-A","CSE-B"],"Prof. Mishra":["CSE-C","IT-A","CS-AI"],"Dr. Kapoor":["IT-B","CS-DS"]}
t_dict["python"]={"Prof. Joshi":["CSE-A","CS-DS"],"Dr. Nair":["CSE-B","CSE-C","IT-A"],"Prof. Bansal":["IT-B","CS-AI"]}
t_dict["cyber"]={"Dr. Saxena":["CSE-A","CSE-C"],"Prof. Tiwari":["CSE-B","IT-B","CS-AI"],"Dr. Malhotra":["IT-A","CS-DS"]}


# Step 16: Assign Teachers


In [ ]:
for s in subs:
    tmp={}
    for tn in t_dict[s]:
        for sec in t_dict[s][tn]:
            tmp[sec]=tn
    df[s+"_t"]=df["class_section"].map(tmp).fillna("Unknown")


# Step 17: Score Teachers for Subjects


In [ ]:
tr=[]
for s in subs:
    for u in range(1,6):
        val=df[s+"_unit_"+str(u)+"_marks"]/m4*100
        for t in df[s+"_t"].unique():
            if t=="Unknown":
                continue
            my_v=val[df[s+"_t"]==t]
            b=(my_v<bad).sum()
            tp=(my_v>=good).sum()
            sc=5+5*tp/len(my_v)-5*b/len(my_v)
            if sc<1:
                sc=1
            if sc>10:
                sc=10
            tr.append({"k1":"sub","k2":s,"k3":"unit "+str(u),"t":t,"sc":sc})


# Step 18: Score Teachers for Labs


In [ ]:
for l in labs:
    val=df["lab_"+l+"_pct"]
    for t in df[l+"_t"].unique():
        if t=="Unknown":
            continue
        my_v=val[df[l+"_t"]==t]
        b=(my_v<bad).sum()
        tp=(my_v>=good).sum()
        sc=5+5*tp/len(my_v)-5*b/len(my_v)
        if sc<1:
            sc=1
        if sc>10:
            sc=10
        tr.append({"k1":"lab","k2":l,"k3":"lab","t":t,"sc":sc})


# Step 19: Rank Best Teachers


In [ ]:
tr_df=pd.DataFrame(tr)
tr_df=tr_df.sort_values("sc",ascending=False)
rk={}
for i,r in tr_df.iterrows():
    key=(r["k1"],r["k2"],r["k3"])
    if key not in rk:
        rk[key]=[]
    rk[key].append((r["t"],r["sc"]))


# Step 20: Setup Variables for Peers


In [ ]:
it=[]
for s in subs:
    it.append(("sub",s,s+"_pct",s+"_bad",s+"_t"))
for l in labs:
    it.append(("lab",l,"lab_"+l+"_pct","lab_"+l+"_bad",l+"_t"))


# Step 21: Find Top Peers


In [ ]:
pdic={}
for x1,x2,x3,x4,x5 in it:
    for sec in df["class_section"].unique():
        ok=df[(df["class_section"]==sec)&(df[x4]==0)]
        top=ok.sort_values(x3,ascending=False).head(5)
        pdic[(x1,x2,sec)]=top[["roll_no","full_name"]].values.tolist()


# Step 22: Assign Mentors to Students


In [ ]:
tn={}
ans=[]
for i,r in df.iterrows():
    for x1,x2,x3,x4,x5 in it:
        if r[x4]==0:
            continue
        wp=""
        rp=""
        if x1=="sub":
            um={}
            for u in range(1,6):
                um[u]=r[x2+"_unit_"+str(u)+"_marks"]
            mk=1
            mv=um[1]
            for uk in um:
                if um[uk]<mv:
                    mv=um[uk]
                    mk=uk
            wp="unit "+str(mk)
            rp=wp
        else:
            es=r["lab_"+x2+"_execution_score"]/m5
            vs=r["lab_"+x2+"_viva_score"]/m6
            if es<vs:
                wp="exe"
            else:
                wp="viva"
            rp="lab"
        ot=r[x5]
        tm="No T"
        ts=0
        for t,s in rk[(x1,x2,rp)]:
            if t!=ot:
                tm=t
                ts=s
                break
        pl=pdic[(x1,x2,r["class_section"])]
        pr="No S"
        pn=""
        if len(pl)>0:
            ky=(x1,x2,r["class_section"])
            if ky not in tn:
                tn[ky]=0
            tn[ky]=tn[ky]+1
            ix=(tn[ky]-1)%len(pl)
            ch=pl[ix]
            pr=ch[0]
            pn=ch[1]
        rs=str(x2)+" "+str(x1)+" bad in "+str(wp)
        ans.append({"roll":r["roll_no"],"name":r["full_name"],"sec":r["class_section"],"gname":r["g_name"],"prio":r["prio"],"bad_tot":r["bad_tot"],"typ":x1,"subj":x2,"pct":round(r[x3],1),"wp":wp,"ot":ot,"tm":tm,"ts":ts,"smr":pr,"smn":pn,"rsn":rs})


# Step 23: Save to CSV


In [ ]:
out=pd.DataFrame(ans)
out=out.sort_values(["prio","bad_tot","pct"],ascending=[True,False,True])
out.to_csv(f_out,index=False)
print("done")
